# Setup
## Dependencies

In [2]:
from pathlib import Path
import csv
import math
import re
import sqlite3

## File Paths

In [3]:
zpt_results_path = Path("../excel/results.csv")
master_db_path = Path("../databases/Master_Database.db").resolve()
master_conn = sqlite3.connect(master_db_path)

# Functions

In [4]:
def get_results(zpt_results_path):
    zpt_output = []

    with open(zpt_results_path, newline="") as results_file:
        for row in csv.DictReader(results_file):
            filename = row["filename"]
            match = re.search(r"-g(\d+)-mosaic\.fits$", filename)

            if match is None:
                print("no ID, skipping:", filename)
                continue
 
            group_id = int(match.group(1))
            zpt_output.append((group_id, filename, row))

    return zpt_output

In [5]:
def check_num(text):
    try:
        return math.isfinite(float(text))
    except ValueError:
        return False

In [6]:
def insert_zeropoints(master_conn, zpt_output, flat=False):
    is_flat = "_f" if flat else ""

    save_sql = f"""
        UPDATE observations
        SET zpt_mean{is_flat} = ?,
            zpt_median{is_flat} = ?,
            zpt_mean_weighted{is_flat} = ?,
            zpt_stdev{is_flat} = ?,
            ztp_err{is_flat} = ?
        WHERE group_id = ?
          AND mosaic_name{is_flat} = ?
          AND zpt_mean{is_flat} IS NULL
          AND error{is_flat} IS NULL
    """
    error_sql = f"""
        UPDATE observations
        SET error{is_flat} = ?
        WHERE group_id = ?
          AND mosaic_name{is_flat} = ?
          AND zpt_mean{is_flat} IS NULL
          AND error{is_flat} IS NULL
    """

    saved = 0
    errors_recorded = 0
    not_updated = 0

    with master_conn:
        for group_id, filename, row in zpt_output:
            mean = row["mean"]
            median = row["median"]
            weighted_mean = row["weighted_mean"]
            stdev = row["stdev"]
            zpt_err = row["zpt_err"]

            check_num_good = True
            for value in [mean, median, weighted_mean, stdev, zpt_err]:
                if not check_num(value):
                    check_num_good = False

            lab_said_done = row["status"] == "done"

            if lab_said_done and check_num_good:
                parameters = (
                    float(mean),
                    float(median),
                    float(weighted_mean),
                    float(stdev),
                    float(zpt_err),
                    group_id,
                    filename,
                )
                cursor = master_conn.execute(save_sql, parameters)

                if cursor.rowcount == 1:
                    saved = saved + 1
                else:
                    not_updated = not_updated + 1

            else:
                message = "zeropoint: " + row["status"]
                parameters = (message, group_id, filename)
                cursor = master_conn.execute(error_sql, parameters)

                if cursor.rowcount == 1:
                    errors_recorded = errors_recorded + 1
                else:
                    not_updated = not_updated + 1
                    
    return saved, errors_recorded, not_updated

# Actual Code Execution

In [7]:
zpt_output = get_results(zpt_results_path)
saved, errors_recorded, not_updated = insert_zeropoints(master_conn, zpt_output, flat=False)

print(f"Saved zeropoints: {saved}")
print(f"Errors recorded: {errors_recorded}")
print(f"Not updated: {not_updated}")

Saved zeropoints: 93
Errors recorded: 2
Not updated: 0
